# BABP 5주차: 미생물의 표현형과 16S rRNA 계통 분석

이번 실습의 중심 질문은 다음과 같습니다.

**“표현형이 비슷한 미생물들은 유전적으로도 가까울까?”**

이를 확인하기 위해 여러 Bacteria와 하나의 Archaea를 함께 준비했습니다. 하지만 어떤 미생물이 Archaea인지는 표현형 matrix를 만들 때 따로 알려주지 않습니다. 눈에 보이는 특징만으로 Archaea를 찾아낼 수 있을지 먼저 예상해봅시다.

또한 실제 이름을 숨긴 `Unknown_X`를 모든 분석에 포함합니다. `Unknown_X`가 표현형 phenogram과 16S rRNA 계통수에서 각각 어디에 놓이는지 관찰하고, 마지막에는 서열 데이터베이스 검색을 통해 정체를 확인해봅시다.

## 0. 환경 설정
16S rRNA 다중서열정렬에 사용할 MUSCLE 5 실행 파일을 준비합니다.

In [ ]:
!pip -q install biopython
!wget -q https://github.com/rcedgar/muscle/releases/download/v5.3/muscle-linux-x86.v5.3 -O muscle
!chmod +x muscle
!./muscle -version

오늘 사용할 라이브러리를 불러옵니다. 각 라이브러리는 데이터 처리, 그림 작성, 계층적 군집화, 서열 분석에 사용됩니다.

In [ ]:
# for matrix processing
import pandas as pd

# for figure plotting
import matplotlib.pyplot as plt
import seaborn as sns

# for hierarchical clustering
from scipy.cluster.hierarchy import linkage, dendrogram

# for sequence download, alignment, and phylogenetic analysis
from Bio import Entrez, SeqIO, AlignIO, Phylo
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor

# for running MUSCLE
import subprocess

sns.set_theme(style="whitegrid")

또한 앞으로 불러올 파일들의 이름을 미리 선언해줍시다. 프로그래밍을 할 때는 파일 이름과 같이 변하지 않는 이름에는 항상 상류에 모아 한 번에 정의해서 관리하기 쉽게 하는 것이 좋습니다.

In [ ]:
CSV_FILE = "phenotypes_data.csv"
UNALIGNED_FASTA = "microbe_16S_sequences.fasta"
ALIGNED_FASTA = "microbe_16S_aligned.fasta"
Entrez.email = "your.email@example.com"  # 자신의 이메일이 아니어도 가능

## 1. Phenogram

먼저 제공된 `phenotypes_data.csv` 파일을 Colab에 업로드합니다. 아래 셀을 실행한 뒤 파일 선택 창에서 CSV 파일을 선택하세요.

이 CSV의 표현형 정보는 미생물 표현형 데이터베이스 **BacDive**에서 수집했습니다. 각 열의 의미는 다음과 같습니다.

- `gram_stain`: Gram staining 결과(`positive`/`negative`)
- `shape`: 세포의 모양(`coccus` 구균, `rod` 간균 등)
- `oxygen_dependence`: 산소 요구성
- `culture_temp`: 최적 배양 온도(°C)
- `access_code`: 16S rRNA 서열을 NCBI에서 내려받기 위한 accession

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
## csv 파일 열어 데이터의 구조 확인하기

phenotype_data = pd.read_csv(CSV_FILE)
display(phenotype_data.drop(columns="access_code"))

**범주형·연속형 자료 변환하기**

컴퓨터는 `positive`, `rod`, `anaerobe` 같은 단어의 생물학적 의미를 스스로 이해하지 못합니다. 미생물 사이의 거리를 계산하려면 이러한 표현형을 비교 가능한 숫자로 변환해야 합니다.

- 그람 염색, 세포 모양, 산소 요구성은 **one-hot encoding**을 적용합니다. 각 범주에 해당하면 1, 해당하지 않으면 0을 부여하여 하나의 특징을 여러 개의 0과 1 열로 펼칩니다.
- 배양 온도는 연속형 자료이며 다른 변수보다 숫자의 크기가 훨씬 크기 때문에 정규화를 진행해야 합니다. **min-max scaling**은 가장 낮은 값을 0, 가장 높은 값을 1로 만드는 정규화 방식입니다.
- 종 이름과 accession code는 미생물의 특징이 아니므로 matrix의 열에 포함하지 않습니다.


In [ ]:
categorical_columns = ["gram_stain", "shape", "oxygen_dependence"]

# 범주형 특징을 0과 1로 변환
categorical_matrix = pd.get_dummies(
    phenotype_data[categorical_columns],
    prefix=categorical_columns,
    dtype=int
)

# 배양 온도를 0~1 범위로 변환
temperature = phenotype_data[["culture_temp"]].astype(float)
temperature_scaled = (temperature - temperature.min()) / (temperature.max() - temperature.min())
temperature_scaled.columns = ["culture_temp_scaled"]

# 두 결과를 하나의 표현형 matrix로 결합
phenotype_matrix = pd.concat([categorical_matrix, temperature_scaled], axis=1)
phenotype_matrix.index = phenotype_data["taxon"]
phenotype_matrix.index.name = "taxon"

display(phenotype_matrix)

**표현형 기반 phenogram 그리기**

변환된 범주형 자료와 배양 온도를 합치면 각 행이 하나의 미생물, 각 열이 하나의 표현형을 나타내는 phenotype matrix가 만들어집니다. 이 matrix로부터, Euclidean distance를 계산하고 average linkage로 계층적 군집화를 수행하여 표현형 기반 **phenogram**을 그려봅시다.

Phenogram에서 가까이 묶인 미생물은 이번에 선택한 표현형 조합이 비슷하다는 의미입니다.
변환한 matrix에서 미생물 사이의 Euclidean distance를 계산하고, 평균연결법(average linkage)으로 계층적 군집화를 수행합니다. 가지가 짧은 위치에서 합쳐질수록 이 실습에서 사용한 표현형이 서로 비슷하다는 뜻입니다.

Phenogram은 관찰한 특징의 유사성을 나타낼 뿐입니다. 그러므로 공통 조상과 진화적 분기 자체를 의미하는 계통수와는 구분해서 생각하도록 합시다.

In [ ]:
phenotype_linkage = linkage(
    phenotype_matrix.values,
    method="average",
    metric="euclidean"
)

plt.figure(figsize=(9, 6))
dendrogram(
    phenotype_linkage,
    labels=phenotype_matrix.index.tolist(),
    orientation="right",
    leaf_font_size=10
)
plt.title("Phenotype-based Phenogram")
plt.xlabel("Euclidean distance")
plt.ylabel("Taxon")
plt.tight_layout()
plt.savefig("phenotype_phenogram.png", dpi=300, bbox_inches="tight")
plt.show()

## 2. Phylogenetic Tree
**NCBI에서 16S rRNA 서열 가져오기**

계통수를 그리기 위해 앞서 설명한 **16S rRNA** 서열을 데이터베이스에서 가져와봅시다.

Week04에서 사용한 `Entrez.efetch()`를 다시 사용합니다. 이번에는 검색어로 accession을 찾는 `esearch()` 과정 없이 CSV에 준비된 accession code로 NCBI Nucleotide 데이터베이스에서 서열을 직접 가져옵니다.

In [ ]:
records = []

for _, row in phenotype_data.iterrows():
    handle = Entrez.efetch(
        db="nucleotide",
        id=row["access_code"],
        rettype="fasta",
        retmode="text"
    )
    record = SeqIO.read(handle, "fasta")
    handle.close()

    # FASTA header를 짧고 일관된 이름으로 변경
    taxon_id = row["taxon"].replace(" ", "_")
    record.id = taxon_id
    record.name = taxon_id
    record.description = ""
    records.append(record)

SeqIO.write(records, UNALIGNED_FASTA, "fasta")
print(f"서열 다운로드 완료: {UNALIGNED_FASTA}")

아래 코드로 다운로드된 서열의 이름과 길이를 확인해봅시다. 아직 정렬하지 않았으므로 서열마다 길이가 조금씩 다를 수 있습니다.

In [ ]:
sequence_summary = pd.DataFrame({
    "taxon": [record.id for record in records],
    "sequence_length": [len(record.seq) for record in records]
})
display(sequence_summary)

**MUSCLE로 다중서열정렬 수행하기**

원본 서열의 같은 위치 번호가 항상 서로 상동인 염기를 의미하지는 않습니다. 진화 과정에서 insertion과 deletion이 일어날 수 있기 때문입니다. 따라서 서열 사이의 차이를 계산하기 전에 **Multiple Sequence Alignment(MSA)**가 필요합니다.

지난 활동에서는 Clustal Omega 웹사이트를 이용했지만, 이번에는 **MUSCLE**을 Colab 환경에 내려받아 Python 코드에서 직접 실행합니다. 정렬 결과에는 필요한 위치에 gap(`-`)이 추가되며, 모든 서열이 같은 정렬 길이를 갖게 됩니다.

MSA가 끝나면 정렬된 각 열을 서로 비교할 수 있습니다. 같은 염기의 비율이 높을수록 두 서열은 가깝고, 다른 염기의 비율이 높을수록 멀다고 판단합니다.

In [ ]:
## MUSCLE 알고리즘으로 MSA 수행하기
subprocess.run(
    ["./muscle", "-align", UNALIGNED_FASTA, "-output", ALIGNED_FASTA],
    check=True
)

print(f"다중서열정렬 완료: {ALIGNED_FASTA}")

In [ ]:
## 정렬된 FASTA를 Biopython으로 읽기
alignment = AlignIO.read(ALIGNED_FASTA, "fasta")

print(f"정렬된 서열 수: {len(alignment)}")
print(f"MSA 전체 길이: {alignment.get_alignment_length()} bp")   # MSA가 끝난 뒤에는 gap(`-`)이 추가되어 모든 서열의 정렬 길이가 같아짐
print("서열 ID:", [record.id for record in alignment])

**16S rRNA 기반 계통수 그리기**

Biopython의 `DistanceCalculator("identity")`를 이용해 정렬된 서열 사이의 distance matrix를 만들고, **Neighbor Joining(NJ)** 알고리즘으로 계통수를 구성합니다. Neighbor Joining은 현재의 거리행렬에서 가까운 대상을 순차적으로 연결하며 전체 가지 길이가 짧은 나무를 찾아가는 distance-based 방법입니다.

이후 `Bio.Phylo`를 이용해 나무를 시각화합니다.

In [ ]:
calculator = DistanceCalculator("identity")
sequence_distance = calculator.get_distance(alignment)

constructor = DistanceTreeConstructor()
phylogenetic_tree = constructor.nj(sequence_distance)
phylogenetic_tree.root_at_midpoint()

fig, ax = plt.subplots(figsize=(11, 7))
Phylo.draw(
    phylogenetic_tree,
    axes=ax,
    do_show=False,
    show_confidence=False,
    label_colors={"Unknown_X": "darkorange"}
)
ax.set_title("16S rRNA Neighbor-Joining Tree")
plt.tight_layout()
plt.savefig("16S_rRNA_phylogenetic_tree.png", dpi=300, bbox_inches="tight")
plt.show()

표현형 phenogram과 16S rRNA 계통수를 나란히 비교해봅시다.

- 표현형에서 가까웠던 미생물이 서열에서도 가까운가요?
- 표현형만 보았을 때 예상한 Archaea는 실제 계통수의 결과와 일치하나요?
- `Unknown_X`는 두 나무에서 각각 어떤 미생물과 가까운가요?

실습 결과, 제공된 표현형 자료에서는 *Escherichia coli*와 *Halobacterium salinarum*이 비교적 비슷하게 묶였습니다. 두 미생물의 Gram staining, 형태, 산소 요구성과 배양 온도가 matrix에서 비슷하게 표현되었기 때문입니다.

하지만 16S rRNA 계통수에서는 Archaea인 *H. salinarum*이 나머지 Bacteria와 뚜렷하게 분리되었습니다. 제한된 표현형만으로는 잘 드러나지 않았던 큰 진화적 차이가 서열에서는 나타난 것입니다. 반대로 표현형만으로는 멀리 떨어져 보였던 미생물이 16S rRNA 계통수에서 더 가까운 관계로 나타나기도 했습니다.

이 결과는 표현형이 의미 없다는 뜻이 아닙니다. 우리가 선택한 몇 가지 표현형만으로는 진화적 역사를 모두 복원하기 어렵고, 계통분류를 위해서는 생물들이 공통 조상으로부터 물려받은 분자적 기록을 함께 살펴봐야 한다는 뜻입니다.

이번 나무 역시 제한된 수의 16S rRNA 서열과 단순한 identity distance로 만든 실습용 결과입니다. 더 신뢰도 높은 계통분석을 위해서는 적절한 evolutionary model, outgroup, bootstrap과 여러 유전자 또는 genome 수준의 정보가 필요할 것입니다.

## 3. Unknown_X의 정체 확인하기

마지막으로 `microbe_16S_sequences.fasta` 파일을 열어 `Unknown_X`의 서열을 복사해봅시다. 이 서열을 NCBI BLAST의 Nucleotide BLAST에 입력하면 데이터베이스에서 가장 유사한 서열을 찾을 수 있습니다.

BLAST 결과를 보기 전에 다음 질문에 먼저 답해보세요.

1. 표현형 phenogram에서는 `Unknown_X`가 어떤 미생물과 가까웠나요?
2. 16S rRNA 계통수에서는 어떤 미생물 가까이에 놓였나요?
3. 세 결과를 바탕으로 `Unknown_X`의 정체를 어떻게 예상할 수 있을까요?
4. BLAST 검색 결과는 자신의 예상과 일치하나요?

정답은 여기에 적지 않겠습니다. 직접 찾는 편이 더 재미있을 테니까요, 한 번 시도해보시길 바랍니다.